# P4-6 · Multi-agent

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q langgraph

## Lý thuyết

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    task: str
    findings: str
    analysis: str
    report: str
    next: str

# worker giả lập để chạy thử đồ thị; trong hệ thật mỗi hàm là một agent có LLM và tool riêng
def researcher(s): print("researcher làm việc"); return {"findings": "3 đối thủ, giá 2–4 triệu"}
def analyst(s):    print("analyst làm việc");    return {"analysis": "giá shop ta ở giữa"}
def writer(s):     print("writer làm việc");     return {"report": "Báo cáo: ..."}

def supervisor(s):
    if not s.get("findings"): return {"next": "researcher"}
    if not s.get("analysis"): return {"next": "analyst"}
    if not s.get("report"): return {"next": "writer"}
    return {"next": "done"}

g = StateGraph(State)
g.add_node("supervisor", supervisor)
g.add_node("researcher", researcher)      # mỗi worker là một agent riêng (có tool riêng)
g.add_node("analyst", analyst)
g.add_node("writer", writer)
g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", lambda s: s["next"],
                        {"researcher": "researcher", "analyst": "analyst", "writer": "writer", "done": END})
for w in ["researcher", "analyst", "writer"]:
    g.add_edge(w, "supervisor")
app = g.compile()
app.invoke({"task": "So sánh giá bàn trang điểm với đối thủ"})["report"]

researcher làm việc
analyst làm việc
writer làm việc
'Báo cáo: ...'


In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class State(TypedDict):
    competitors: list[str]
    results: Annotated[list, operator.add]        # mỗi worker thêm phần của mình
    report: str

class WorkerInput(TypedDict):
    name: str

def fan_out(s: State):
    return [Send("research_one", {"name": c}) for c in s["competitors"]]

def research_agent(name):                          # giả lập; hệ thật: một agent nhỏ có tool tìm kiếm
    return f"{name}: giá từ {len(name)} triệu"

def writer_llm(results):                           # giả lập; hệ thật: một lần gọi LLM tổng hợp
    return " | ".join(r["summary"] for r in results)

def research_one(w: WorkerInput):
    summary = research_agent(w["name"])            # một agent nhỏ, context riêng
    return {"results": [{"name": w["name"], "summary": summary}]}

def write(s: State):
    return {"report": writer_llm(s["results"])}

g = StateGraph(State)
g.add_node("research_one", research_one)
g.add_node("write", write)
g.add_conditional_edges(START, fan_out, ["research_one"])
g.add_edge("research_one", "write")
g.add_edge("write", END)
app = g.compile()
app.invoke({"competitors": ["Hòa Bình", "An Phát", "Gỗ Việt"], "results": []})

{'competitors': ['Hòa Bình', 'An Phát', 'Gỗ Việt'], 'results': [{'name': 'Hòa Bình', 'summary': 'Hòa Bình: giá từ 8 triệu'}, {'name': 'An Phát', 'summary': 'An Phát: giá từ 7 triệu'}, {'name': 'Gỗ Việt', 'summary': 'Gỗ Việt: giá từ 7 triệu'}], 'report': 'Hòa Bình: giá từ 8 triệu | An Phát: giá từ 7 triệu | Gỗ Việt: giá từ 7 triệu'}


## Bài tập 5: Fan-out song song {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class State(TypedDict):
    competitors: list[str]
    findings: Annotated[list, operator.add]     # các nhánh song song cùng ghi vào, nối danh sách lại
    report: str

def fan_out(state):
    return [Send("research_one", {"name": c}) for c in state["competitors"]]

def research_one(arg):
    summary = research_agent(arg["name"])       # agent con với tool tìm kiếm, trả bản tóm tắt
    return {"findings": [{"name": arg["name"], "summary": summary}]}

def write(state):
    return {"report": writer_agent(state["findings"])}

g = StateGraph(State)
g.add_node("research_one", research_one)
g.add_node("write", write)
g.add_conditional_edges(START, fan_out, ["research_one"])
g.add_edge("research_one", "write")
g.add_edge("write", END)
app = g.compile()